### Initial Setup

In [21]:
# CONFIG
COCO_JSON_PATH = "/content/instances_COCO.json"
IMAGES_DIR = "/content"
OUTPUT_DIR = "/content/output_white_bg"

In [22]:
!pip install opencv-python numpy pycocotools
from IPython.display import clear_output
clear_output()

In [23]:
import os
import cv2
import numpy as np
import json
from pycocotools import mask as maskUtils

### I / O Helpers

In [24]:
def load_coco(json_path: str) -> dict:
    with open(json_path) as f:
        return json.load(f)

In [25]:
def build_index(coco: dict) -> tuple[dict, dict]:
    images_dict     = {img["id"]: img for img in coco["images"]}
    categories_dict = {cat["id"]: cat["name"] for cat in coco["categories"]}
    return images_dict, categories_dict

In [26]:
def setup_output_dirs(output_dir: str, categories_dict: dict) -> None:
    for cat_name in categories_dict.values():
        os.makedirs(os.path.join(output_dir, cat_name), exist_ok=True)

### Mask builders

In [27]:
def mask_from_rle(rle: dict, h: int, w: int) -> np.ndarray:
    if isinstance(rle["counts"], list):
        rle = maskUtils.frPyObjects(rle, h, w)
    return maskUtils.decode(rle).astype(np.uint8) * 255

In [28]:
def mask_from_bbox(bbox: list, h: int, w: int) -> np.ndarray:
    mask = np.zeros((h, w), dtype=np.uint8)
    x, y, bw, bh = map(int, bbox)
    cv2.rectangle(mask, (x, y), (x + bw, y + bh), 255, -1)
    return mask

In [29]:
def _draw_segment(mask: np.ndarray, seg: list | dict, h: int, w: int, shape_type: str) -> None:
    """Draw a single segment (polygon/rect/ellipse/RLE) onto mask in-place."""
    if isinstance(seg, dict):
        decoded = mask_from_rle(seg, h, w)
        np.maximum(mask, decoded, out=mask)
        return

    flat = np.array(seg, dtype=np.float32)
    pts  = flat.reshape(-1, 2)

    if len(flat) == 5 or shape_type == "ellipse":
        try:
            cx, cy, rx, ry, angle = flat[:5]
            cv2.ellipse(mask, (int(cx), int(cy)),
                        (int(abs(rx)), int(abs(ry))), float(angle),
                        0, 360, 255, -1)
        except Exception as e:
            print(f"⚠️  Ellipse draw failed: {e}")

    elif len(pts) == 2 or (len(flat) == 4 and shape_type in ("rectangle", "points")):
        x1, y1 = int(pts[0][0]), int(pts[0][1])
        x2, y2 = int(pts[1][0]), int(pts[1][1])
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)

    elif len(pts) >= 3:
        cv2.fillPoly(mask, [pts.astype(np.int32)], 255)

    else:
        print(f"⚠️  Unrecognised segment shape: {len(flat)} values")

In [30]:
def build_mask(ann: dict, h: int, w: int) -> np.ndarray:
    mask       = np.zeros((h, w), dtype=np.uint8)
    seg        = ann.get("segmentation")
    shape_type = ann.get("attributes", {}).get("type", "polygon")

    if isinstance(seg, list) and seg:
        for s in seg:
            _draw_segment(mask, s, h, w, shape_type)

    elif isinstance(seg, dict):
        mask = mask_from_rle(seg, h, w)

    else:
        print(f"ℹ️  No segmentation for ann {ann['id']}, falling back to bbox")
        mask = mask_from_bbox(ann["bbox"], h, w)

    return mask

### Apply mask & save

In [31]:
def apply_and_save(img: np.ndarray, mask: np.ndarray,
                   out_path: str) -> bool:
    if mask.max() == 0:
        return False

    white  = np.ones_like(img) * 255
    result = np.where(mask[:, :, None] == 255, img, white)

    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y, bw, bh = cv2.boundingRect(coords)
        result = result[y:y + bh, x:x + bw]

    cv2.imwrite(out_path, result)
    return True

### Pipeline

In [32]:
def extract_annotations(coco_json_path: str, images_dir: str, output_dir: str) -> None:
    coco                        = load_coco(coco_json_path)
    images_dict, categories_dict = build_index(coco)
    setup_output_dirs(output_dir, categories_dict)

    for ann in coco["annotations"]:
        img_info = images_dict[ann["image_id"]]
        img_path = os.path.join(images_dir, img_info["file_name"])

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️  Could not read image: {img_path}")
            continue

        h, w = img.shape[:2]
        mask = build_mask(ann, h, w)

        cat_name = categories_dict.get(ann["category_id"], "unknown")
        out_path = os.path.join(output_dir, cat_name, f"{cat_name}__ann{ann['id']}.png")

        if not apply_and_save(img, mask, out_path):
            print(f"⚠️  Empty mask for ann {ann['id']} — skipping")

    print("✅ Extraction complete!")

In [33]:
extract_annotations(
      coco_json_path=COCO_JSON_PATH,
      images_dir=IMAGES_DIR,
      output_dir=OUTPUT_DIR,
  )

ℹ️  No segmentation for ann 4, falling back to bbox
ℹ️  No segmentation for ann 9, falling back to bbox
ℹ️  No segmentation for ann 19, falling back to bbox
ℹ️  No segmentation for ann 21, falling back to bbox
ℹ️  No segmentation for ann 24, falling back to bbox
ℹ️  No segmentation for ann 25, falling back to bbox
ℹ️  No segmentation for ann 26, falling back to bbox
ℹ️  No segmentation for ann 27, falling back to bbox
ℹ️  No segmentation for ann 28, falling back to bbox
ℹ️  No segmentation for ann 111, falling back to bbox
ℹ️  No segmentation for ann 112, falling back to bbox
ℹ️  No segmentation for ann 113, falling back to bbox
ℹ️  No segmentation for ann 117, falling back to bbox
ℹ️  No segmentation for ann 124, falling back to bbox
ℹ️  No segmentation for ann 142, falling back to bbox
ℹ️  No segmentation for ann 143, falling back to bbox
ℹ️  No segmentation for ann 144, falling back to bbox
ℹ️  No segmentation for ann 146, falling back to bbox
ℹ️  No segmentation for ann 147, fallin

In [ ]:
# !zip -r output_white_bg.zip output_white_bg/
# from google.colab import files
# files.download(f"{OUTPUT_DIR}.zip")

In [20]:
# !rm -rf /content/output_white_bg